In [51]:
# ============================================================
# Cell 1: configuration and functions (run once; re-run after edits)
# ============================================================
import json
import concurrent.futures

import numpy as np
import awkward as ak
import pyarrow.dataset as ds

# ---------------- Analysis configuration: keep in sync with the datacard ----------------
GEN_PT_BINS = [0, 200, 300, 450]       # signal split on genZHpt: [0,200), [200,300), [300,450), >=450
MASS_BINS   = [50, 80, 105, 145, 200]
BBVL_WP     = 0.9172                   # datacard: ZH_bbvLscore >= 0.9172

# Reco channels, right-inclusive like pd.cut in the datacard: (200,300], (300,450], (450,inf)
CHANNELS = {"Zhpt1": (200, 300), "Zhpt2": (300, 450), "Zhpt3": (450, np.inf)}
MERGE_LAST_MASS_BIN = {"Zhpt1"}        # datacard merges the last two mass bins in Zhpt1 only
# NOTE: follows the datacard with isblind=True (full flatten). Unblinding changes the unrolling.

# Discriminant: same as the pickling script, disc = sig / (sig + ttbb + ttlf)
SPANET_MODEL = "noOverlap_MultiWP"
SCORE_COLUMNS = [f"events_{s}_score_{SPANET_MODEL}" for s in ("sig", "ttbb", "ttlf")]
NN_BINS = {   # edges from DataCardShapes for disc_<SPANET_MODEL>
    "Zhpt1": [0.0, 0.11901899, 0.27811474, 0.36523486, 0.54510037, 0.74072657, 1.0],
    "Zhpt2": [0.0, 0.18853479, 0.39141932, 0.48251952, 0.69055395, 0.85575131, 1.0],
    "Zhpt3": [0.0, 0.12718003, 0.3305542, 0.42866625, 0.61717568, 0.8092443, 1.0],
}

BASE = "/cms/data/store/user/jsamudio/eftchunks/2018"
PROCESSES = {
    # decay_mode: "all" (no filter), "bb" (flag True only), "nonbb" (flag False only)
    "ttH": {"paths": [f"{BASE}/ttHSMEFT_2018/btag_mask/nominal"],
            "gen_split": True, "decay_flag": "events_Hbb", "decay_mode": "all"},
    "ttZ": {"paths": [f"{BASE}/ttZSMEFT_2018/btag_mask/nominal"],
            "gen_split": True, "decay_flag": "events_Zbb", "decay_mode": "all"},
    # Add tt_B back here (tt+B subsample output) if EFT effects on ttbb are wanted:
    "tt_B": {"paths": [f"{BASE}/ttbbSMEFT_2018/btag_mask/nominal"],
             "gen_split": False, "decay_flag": None, "decay_mode": "all"},
}

BASE_COLUMNS = [
    "events_EFTfitCoefficients", "events_EFTfitCoefficientIndex1",
    "events_EFTfitCoefficientIndex2", "events_WCnames",
    "events_n_ak4jets", "events_n_b_outZH", "events_ZH_bbvLscore",
    "events_MET_pt", "events_ZH_M", "events_ZH_pt", *SCORE_COLUMNS,
]


# ---------------- Helpers ----------------
def decode_WCnames(WCnames):
    out, name = [], ""
    for val in WCnames:
        four = int(val).to_bytes(4, "big").decode()
        if "-" in four:
            name += four.split("-")[-1]
        else:
            if name:
                out.append(name)
            name = four.lstrip("\x00")
    out.append(name)
    return out


def bin_index(values, edges):
    """Bin like np.histogram: left-inclusive, last bin also includes the upper edge."""
    edges = np.asarray(edges, dtype=float)
    idx = np.searchsorted(edges, values, side="right") - 1
    idx[values == edges[-1]] = len(edges) - 2
    return idx


def fill(acc, nn, mass, coefs, nn_edges, mass_edges):
    i, j = bin_index(nn, nn_edges), bin_index(mass, mass_edges)
    ok = (i >= 0) & (i < len(nn_edges) - 1) & (j >= 0) & (j < len(mass_edges) - 1)
    np.add.at(acc, (i[ok], j[ok]), coefs[ok])


def gen_masks(label, gen_split, genpt, n):
    if not gen_split:
        return {label: np.ones(n, dtype=bool)}
    edges = GEN_PT_BINS + [np.inf]
    return {f"{label}{i}": (genpt >= edges[i]) & (genpt < edges[i + 1])
            for i in range(len(GEN_PT_BINS))}


def unroll(w, channel):
    """Same layout as the datacard: (nn, mass) flattened row-major, mass fastest."""
    if channel in MERGE_LAST_MASS_BIN:
        w = w.copy()
        w[:, -2, :] += w[:, -1, :]
        w = w[:, :-1, :]
    return w.reshape(-1, w.shape[-1])

DIAGONAL = {"Zhpt1": 1, "Zhpt2": 2, "Zhpt3": 3}   # reco channel -> matching gen-pT bin index

def is_diagonal(channel, process):
    """True if a gen-pT-split signal process matches the reco channel (e.g. Zhpt3/ttH3)."""
    i = DIAGONAL.get(channel)
    return i is not None and process[-1:].isdigit() and int(process[-1]) == i


# ---------------- Reading the EFT samples ----------------
def process_path(label, cfg, path):
    """Accumulate binned EFT coefficients (+ raw count and SM weight^2) for one directory."""
    print(f"[{label}] reading {path}")
    columns = list(BASE_COLUMNS)
    if cfg["gen_split"]:
        columns.append("events_genZHpt")
    use_flag = cfg["decay_mode"] != "all"
    if use_flag:
        columns.append(cfg["decay_flag"])

    acc, meta, k00 = {}, None, None
    for batch in ds.dataset(path, format="parquet").to_batches(columns=columns):
        chunk = ak.from_arrow(batch)
        if len(chunk) == 0:
            continue
        if meta is None:
            idx1 = ak.to_numpy(chunk["events_EFTfitCoefficientIndex1"][0]).astype(int)
            idx2 = ak.to_numpy(chunk["events_EFTfitCoefficientIndex2"][0]).astype(int)
            wc_names = ["SM"] + decode_WCnames(ak.to_list(chunk["events_WCnames"][0]))
            meta = (idx1, idx2, wc_names)
            k00 = int(np.flatnonzero((idx1 == 0) & (idx2 == 0))[0])

        col = lambda name: ak.to_numpy(chunk[name])
        coefs = col("events_EFTfitCoefficients").astype(np.float64)
        # Extra columns: raw event count and squared SM weight (SM stat. uncertainty)
        coefs = np.column_stack([coefs, np.ones(len(coefs)), coefs[:, k00] ** 2])

        zh_pt, zh_m = col("events_ZH_pt"), col("events_ZH_M")
        s_sig, s_ttbb, s_ttlf = (col(c).astype(np.float64) for c in SCORE_COLUMNS)
        denom = s_sig + s_ttbb + s_ttlf
        nn = np.where(denom > 0, s_sig / np.where(denom > 0, denom, 1.0), 0.0)

        sel = ((col("events_n_ak4jets") >= 5) & (col("events_n_b_outZH") == 2) &
               (col("events_ZH_bbvLscore") >= BBVL_WP) & (col("events_MET_pt") >= 20) &
               (zh_m >= 50) & (zh_m <= 200) & (nn >= 0.0))
        if use_flag:
            flag = col(cfg["decay_flag"]).astype(bool)
            sel &= flag if cfg["decay_mode"] == "bb" else ~flag

        genpt = col("events_genZHpt") if cfg["gen_split"] else None
        gmasks = gen_masks(label, cfg["gen_split"], genpt, len(sel))

        for ch, (lo, hi) in CHANNELS.items():
            ch_sel = sel & (zh_pt > lo) & (zh_pt <= hi)
            for proc, gmask in gmasks.items():
                key = (ch, proc)
                if key not in acc:
                    acc[key] = np.zeros((len(NN_BINS[ch]) - 1, len(MASS_BINS) - 1, coefs.shape[1]))
                m = ch_sel & gmask
                if m.any():
                    fill(acc[key], nn[m], zh_m[m], coefs[m], NN_BINS[ch], MASS_BINS)
    return acc, meta


def merge_results(results):
    """Sum binned coefficients across inputs per (channel, process)."""
    merged, meta = {}, {}
    for acc, m in results:
        if m is None:
            continue
        for key, w in acc.items():
            if key in merged:
                prev = meta[key]
                if (prev[2] != m[2] or not np.array_equal(prev[0], m[0])
                        or not np.array_equal(prev[1], m[1])):
                    raise ValueError(f"Coefficient layout differs between inputs for {key}")
                merged[key] += w
            else:
                merged[key], meta[key] = w.copy(), m
    return merged, meta


def read_inputs(processes=None, n_workers=4):
    """Read all EFT samples once. Returns (merged, meta) for build_scaling().
    n_workers=1 runs serially (use it if multiprocessing misbehaves in your kernel)."""
    processes = processes or PROCESSES
    jobs = [(label, cfg, path) for label, cfg in processes.items() for path in cfg["paths"]]
    if n_workers > 1:
        with concurrent.futures.ProcessPoolExecutor(max_workers=n_workers) as ex:
            results = list(ex.map(process_path, *zip(*jobs)))
    else:
        results = [process_path(*job) for job in jobs]
    return merge_results(results)


# ---------------- Building and checking the Combine entries ----------------
def combine_parameters(wc_names, wcs=None, wc_range=(-20, 20)):
    lo, hi = wc_range
    wanted = wc_names[1:] if wcs is None else wcs
    params = ["SM[1]"] + [f"{wc}[0,{lo},{hi}]" for wc in wanted if wc in wc_names]
    missing = [wc for wc in wanted if wc not in wc_names]
    return params, missing


def build_entry(unrolled, idx1, idx2, wc_names, params, channel, process,
                coef_threshold=0.0, log=print):
    sel = np.array([0 if p.split("[")[0] == "SM" else wc_names.index(p.split("[")[0])
                    for p in params])
    nbins, ncoef, n_nano = unrolled.shape[0], len(params), len(wc_names)

    # Symmetric matrix: coefficient of c_a*c_b = 2*M_ab for a != b
    M_full = np.zeros((nbins, n_nano, n_nano))
    for k, (a, b) in enumerate(zip(idx1, idx2)):
        if a == b:
            M_full[:, a, a] = unrolled[:, k]
        else:
            M_full[:, a, b] = M_full[:, b, a] = 0.5 * unrolled[:, k]

    M = M_full[:, sel][:, :, sel]
    sm = M[:, 0, 0].copy()
    good = sm > 0
    if np.any(~good):
        log(f"  [{channel}/{process}] {np.sum(~good)} bins with SM yield <= 0 -> pure SM scaling")
    M[good] /= sm[good][:, None, None]
    M[~good] = 0.0
    M[~good, 0, 0] = 1.0

    rows, cols = np.tril_indices(ncoef)
    scaling = M[:, rows, cols]
    if coef_threshold > 0:
        scaling[np.abs(scaling) < coef_threshold] = 0.0
    entry = {"channel": channel, "process": process,
             "parameters": params, "scaling": scaling.tolist()}
    return entry, sm, good


def eval_scaling(entry, point):
    """Per-bin EFT/SM ratio of an entry at a WC point, e.g. {'ctgre': 0.5}."""
    names = [p.split("[")[0] for p in entry["parameters"]]
    theta = np.array([1.0 if n == "SM" else point.get(n, 0.0) for n in names])
    rows, cols = np.tril_indices(len(theta))
    weights = np.where(rows == cols, 1.0, 2.0) * theta[rows] * theta[cols]
    return np.asarray(entry["scaling"]) @ weights


def closure_check(entry, unrolled, idx1, idx2, wc_names, sm, good, point):
    """Direct polynomial sum vs. SM * (theta^T M theta); should agree to ~1e-12."""
    kept = {p.split("[")[0] for p in entry["parameters"]}
    c = np.array([1.0] + [point.get(n, 0.0) if n in kept else 0.0 for n in wc_names[1:]])
    direct = unrolled @ (c[idx1] * c[idx2])
    predicted = sm * eval_scaling(entry, point)
    if not np.any(good):
        return 0.0
    rel = np.abs(predicted[good] - direct[good]) / np.maximum(np.abs(direct[good]), 1e-30)
    return float(np.max(rel))

def bin_labels(channel):
    """(nn_index, mass_index, nn_range, mass_range) for each unrolled bin, in datacard order."""
    nn = NN_BINS[channel]
    mass = list(MASS_BINS)
    if channel in MERGE_LAST_MASS_BIN:          # last two mass bins merged: drop the shared edge
        mass = mass[:-2] + [mass[-1]]
    labels = []
    for i in range(len(nn) - 1):
        for j in range(len(mass) - 1):
            labels.append((i, j, f"[{nn[i]:.3f}, {nn[i+1]:.3f}]", f"[{mass[j]:g}, {mass[j+1]:g}]"))
    return labels, nn, mass


def flag_reasons(n_raw, sm, sm2, min_events, max_rel_err):
    """Text reason per bin for failing the low-statistics cuts ('' if it passes)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        rel_err = np.where(sm > 0, np.sqrt(sm2) / sm, np.inf)
    reasons = []
    for n, s, r in zip(n_raw, sm, rel_err):
        why = []
        if n < min_events:
            why.append(f"n<{min_events}")
        if s <= 0:
            why.append("SM<=0")
        elif r > max_rel_err:
            why.append(f"rel.err>{max_rel_err:.0%}")
        reasons.append(", ".join(why))
    return reasons, rel_err


def print_flag_grid(channel, flags, log=print):
    """NN x mass map of flagged bins: X = smoothed, . = own ratio."""
    labels, nn, mass = bin_labels(channel)
    n_mass = len(mass) - 1
    header = "".join(f"{f'{mass[j]:g}-{mass[j+1]:g}':>10}" for j in range(n_mass))
    log(f"      {'NN // mass [GeV]':>22}{header}")
    for i in range(len(nn) - 1):
        row = "".join(f"{('X' if flags[i * n_mass + j] else '.'):>10}" for j in range(n_mass))
        log(f"      {f'[{nn[i]:.3f}, {nn[i+1]:.3f}]':>22}{row}")


def unreliable(n_raw, sm, sm2, min_events, max_rel_err):
    """True where the EFT-sample statistics are too poor for a per-bin ratio."""
    with np.errstate(divide="ignore", invalid="ignore"):
        rel_err = np.where(sm > 0, np.sqrt(sm2) / sm, np.inf)
    return (n_raw < min_events) | (sm <= 0) | (rel_err > max_rel_err)

def bin_labels(channel):
    """(nn_index, mass_index, nn_range, mass_range) for each unrolled bin, in datacard order."""
    nn = NN_BINS[channel]
    mass = list(MASS_BINS)
    if channel in MERGE_LAST_MASS_BIN:          # last two mass bins merged: drop the shared edge
        mass = mass[:-2] + [mass[-1]]
    labels = []
    for i in range(len(nn) - 1):
        for j in range(len(mass) - 1):
            labels.append((i, j, f"[{nn[i]:.3f}, {nn[i+1]:.3f}]", f"[{mass[j]:g}, {mass[j+1]:g}]"))
    return labels, nn, mass


def flag_reasons(n_raw, sm, sm2, min_events, max_rel_err):
    """Text reason per bin for failing the low-statistics cuts ('' if it passes)."""
    with np.errstate(divide="ignore", invalid="ignore"):
        rel_err = np.where(sm > 0, np.sqrt(sm2) / sm, np.inf)
    reasons = []
    for n, s, r in zip(n_raw, sm, rel_err):
        why = []
        if n < min_events:
            why.append(f"n<{min_events}")
        if s <= 0:
            why.append("SM<=0")
        elif r > max_rel_err:
            why.append(f"rel.err>{max_rel_err:.0%}")
        reasons.append(", ".join(why))
    return reasons, rel_err


def print_flag_grid(channel, flags, log=print):
    """NN x mass map of flagged bins: X = smoothed, . = own ratio."""
    labels, nn, mass = bin_labels(channel)
    n_mass = len(mass) - 1
    header = "".join(f"{f'{mass[j]:g}-{mass[j+1]:g}':>10}" for j in range(n_mass))
    log(f"      {'NN // mass [GeV]':>22}{header}")
    for i in range(len(nn) - 1):
        row = "".join(f"{('X' if flags[i * n_mass + j] else '.'):>10}" for j in range(n_mass))
        log(f"      {f'[{nn[i]:.3f}, {nn[i+1]:.3f}]':>22}{row}")

BOSON_MASS = {"ttH": 125.0, "ttZ": 91.19}

def diagonal_mask(channel, process):
    """Bins of a diagonal entry (gen-pT bin == reco channel) in the boson's mass-peak bin."""
    labels, nn, mass = bin_labels(channel)
    if not is_diagonal(channel, process):
        return np.zeros(len(labels), dtype=bool)
    m0 = BOSON_MASS.get(process.rstrip("0123456789"))
    if m0 is None:
        return np.zeros(len(labels), dtype=bool)
    j_peak = int(np.searchsorted(mass, m0, side="right") - 1)
    return np.array([j == j_peak for (_, j, _, _) in labels])


def build_scaling(merged, meta, output="combine_scaling_2018.json", fallback="auto",
                  min_events=20, max_rel_err=0.3, wcs=None, wc_range=(-20, 20),
                  coef_threshold=0.0, verbose=True, show_bins="grid", keep_diagonal=False):
    """
    Build the interferenceModel scaling JSON from the output of read_inputs().

    fallback : "auto" - bins failing the cuts use the (channel, process) ratio, or the
                        process ratio if the channel total also fails
               "none" - every bin uses its own ratio (original behaviour)
               "all"  - every bin uses the channel-level (or process-level) ratio
    min_events, max_rel_err : low-statistics cuts on the EFT sample in each bin
    wcs      : None = every WC in the samples, or a list to restrict
    Returns the list of entries (also written to `output`).
    """
    if fallback not in ("auto", "none", "all"):
        raise ValueError("fallback must be 'auto', 'none' or 'all'")
    log = print if verbose else (lambda *a, **k: None)

    # Process-level totals (all channels and bins), the last-resort fallback
    proc_totals = {}
    for (ch, proc), full in merged.items():
        ncoef = len(meta[(ch, proc)][0])
        proc_totals[proc] = proc_totals.get(proc, 0) + full[..., :ncoef].sum(axis=(0, 1))

    entries, n_flagged, n_bins_total, worst_closure = [], 0, 0, 0.0
    for (ch, proc) in sorted(merged):
        full = merged[(ch, proc)]
        idx1, idx2, wc_names = meta[(ch, proc)]
        ncoef = len(idx1)
        k00 = int(np.flatnonzero((idx1 == 0) & (idx2 == 0))[0])
        if not np.any(full[..., :ncoef]):
            log(f"  [{ch}/{proc}] empty, skipped")
            continue
        params, missing = combine_parameters(wc_names, wcs, wc_range)
        if missing:
            log(f"  [{ch}/{proc}] WCs not in sample (omitted): {missing}")

        unrolled_full = unroll(full, ch)
        unrolled = unrolled_full[:, :ncoef].copy()
        n_raw = unrolled_full[:, ncoef].copy()
        sm_raw = unrolled[:, k00].copy()
        sm2 = unrolled_full[:, ncoef + 1].copy()

        if fallback == "none":
            low_stat = np.zeros(len(n_raw), dtype=bool)
        elif fallback == "all":
            low_stat = np.ones(len(n_raw), dtype=bool)
        else:
            low_stat = unreliable(n_raw, sm_raw, sm2, min_events, max_rel_err)

        if keep_diagonal:
            keep = diagonal_mask(ch, proc)
            n_kept = int(np.sum(low_stat & keep))
            if n_kept:
                log(f"  [{ch}/{proc}] keeping own ratio in {n_kept} diagonal mass-peak "
                    f"bins that would otherwise be smoothed")
            low_stat = low_stat & ~keep

        if np.any(low_stat):
            ch_tot = unrolled.sum(axis=0)
            ch_ok = not unreliable(np.array([n_raw.sum()]), np.array([ch_tot[k00]]),
                                   np.array([sm2.sum()]), min_events, max_rel_err)[0]
            replacement, level = (ch_tot, "channel-level") if ch_ok else (proc_totals[proc], "process-level")
            unrolled[low_stat] = replacement   # normalization cancels in the per-bin ratio
            log(f"  [{ch}/{proc}] {np.sum(low_stat)}/{len(low_stat)} bins smoothed -> {level} ratio")

            if show_bins == "grid":
                print_flag_grid(ch, low_stat, log)
            elif show_bins == "list":
                labels, _, _ = bin_labels(ch)
                reasons, rel_err = flag_reasons(n_raw, sm_raw, sm2, min_events, max_rel_err)
                for b in np.flatnonzero(low_stat):
                    i, j, nn_rng, m_rng = labels[b]
                    why = reasons[b] or f"forced (fallback='{fallback}')"
                    log(f"      bin {b:2d}: NN bin {i} {nn_rng}, mass bin {j} {m_rng} GeV, "
                        f"raw={int(n_raw[b])}, rel.err={rel_err[b]:.2f}  [{why}]")
        n_flagged += int(np.sum(low_stat))
        n_bins_total += len(low_stat)

        entry, sm, good = build_entry(unrolled, idx1, idx2, wc_names, params, ch, proc,
                                      coef_threshold, log)
        point = {p.split("[")[0]: 1.0 for p in params[1:]}
        rel = closure_check(entry, unrolled, idx1, idx2, wc_names, sm, good, point)
        worst_closure = max(worst_closure, rel)
        status = "OK" if rel < 1e-8 or coef_threshold > 0 else "CHECK!"
        log(f"  [{ch}/{proc}] bins={unrolled.shape[0]}, raw events={int(n_raw.sum())}, "
            f"SM sum={sm_raw.sum():.4g}, closure max rel. diff={rel:.2e} {status}")
        entries.append(entry)

    with open(output, "w") as f:
        json.dump(entries, f, indent=1)
    print(f"Wrote {len(entries)} entries to {output}  (fallback='{fallback}', "
          f"keep_diagonal={keep_diagonal}, flagged {n_flagged}/{n_bins_total} bins, "
          f"worst closure {worst_closure:.1e})")

    if verbose:
        wcs_by_proc = {}
        for e in entries:
            base = e["process"].rstrip("0123456789")
            wcs_by_proc.setdefault(base, [p.split("[")[0] for p in e["parameters"][1:]])
        for base, names in wcs_by_proc.items():
            print(f"  {base}: {len(names)} WCs -> {', '.join(names)}")
    return entries


In [54]:
entries = build_scaling(merged, meta, output="scaling_keepdiag.json", keep_diagonal=True)

  [Zhpt1/ttH0] bins=18, raw events=5378, SM sum=8326, closure max rel. diff=1.04e-15 OK
  [Zhpt1/ttH1] bins=18, raw events=4026, SM sum=3350, closure max rel. diff=4.73e-16 OK
  [Zhpt1/ttH2] 6/18 bins smoothed -> channel-level ratio
            NN // mass [GeV]     50-80    80-105   105-200
              [0.000, 0.119]         X         .         X
              [0.119, 0.278]         .         .         .
              [0.278, 0.365]         X         .         .
              [0.365, 0.545]         X         .         .
              [0.545, 0.741]         X         .         .
              [0.741, 1.000]         X         .         .
  [Zhpt1/ttH2] bins=18, raw events=3411, SM sum=1505, closure max rel. diff=1.00e-15 OK
  [Zhpt1/ttH3] 18/18 bins smoothed -> channel-level ratio
            NN // mass [GeV]     50-80    80-105   105-200
              [0.000, 0.119]         X         X         X
              [0.119, 0.278]         X         X         X
              [0.278, 0.365]   

In [53]:
# ============================================================
# Cell 2: run
# ============================================================
# Read all EFT samples once (the slow part). Re-run only if the samples or
# the selection/binning in Cell 1 change.
merged, meta = read_inputs(n_workers=4)

# Nominal scaling file
#entries = build_scaling(merged, meta, output="combine_scaling_2018.json")                    # grids
#entries = build_scaling(merged, meta, output="combine_scaling_2018.json", show_bins="list")  # details

# Low-stat cross-check variants (fast: reuses the inputs read above)
# variants = {
#     "nofallback": dict(fallback="none"),
#     "pooled":     dict(fallback="all"),
#     "loose":      dict(min_events=10, max_rel_err=0.5),
#     "tight":      dict(min_events=50, max_rel_err=0.2),
# }
# for name, opts in variants.items():
#     build_scaling(merged, meta, output=f"scaling_{name}.json", verbose=False, **opts)


[ttZ] reading /cms/data/store/user/jsamudio/eftchunks/2018/ttZSMEFT_2018/btag_mask/nominal[ttH] reading /cms/data/store/user/jsamudio/eftchunks/2018/ttHSMEFT_2018/btag_mask/nominal[tt_B] reading /cms/data/store/user/jsamudio/eftchunks/2018/ttbbSMEFT_2018/btag_mask/nominal




In [35]:
import uproot

def pooled_yield_fraction(merged, meta, rootfile, min_events=20, max_rel_err=0.3):
    f = uproot.open(rootfile)
    totals = {}
    for (ch, proc) in sorted(merged):
        full = merged[(ch, proc)]
        idx1, idx2, _ = meta[(ch, proc)]
        n = len(idx1)
        k00 = int(np.flatnonzero((idx1 == 0) & (idx2 == 0))[0])
        u = unroll(full, ch)
        flags = unreliable(u[:, n], u[:, k00], u[:, n + 1], min_events, max_rel_err)
        try:
            y = f[f"{ch}_{proc}"].values()
        except KeyError:
            continue
        frac = y[flags].sum() / y.sum() if y.sum() > 0 else 0.0
        print(f"{ch}/{proc:5}: {flags.sum():2d}/{len(flags)} bins, {frac:6.1%} of template yield smoothed")
        base = proc.rstrip("0123456789")
        acc = totals.setdefault(base, [0.0, 0.0])
        acc[0] += y[flags].sum()
        acc[1] += y.sum()
    for base, (pooled, total) in totals.items():
        print(f"TOTAL {base}: {pooled / total:.1%} of expected yield uses a pooled ratio")

pooled_yield_fraction(merged, meta,
    "datacards/datacard_nominal2018_disc_overlapRemoval_MultiWP_eftTTB_2018.root")

Zhpt1/ttH0 :  0/18 bins,   0.0% of template yield smoothed
Zhpt1/ttH1 :  0/18 bins,   0.0% of template yield smoothed
Zhpt1/ttH2 :  4/18 bins,   6.2% of template yield smoothed
Zhpt1/ttH3 : 17/18 bins,  92.4% of template yield smoothed
Zhpt1/ttZ0 : 11/18 bins,  39.5% of template yield smoothed
Zhpt1/ttZ1 :  6/18 bins,   7.2% of template yield smoothed
Zhpt1/ttZ2 : 14/18 bins,  46.4% of template yield smoothed
Zhpt1/ttZ3 : 18/18 bins, 100.0% of template yield smoothed
Zhpt1/tt_B :  0/18 bins,   0.0% of template yield smoothed
Zhpt2/ttH0 :  4/24 bins,   5.2% of template yield smoothed
Zhpt2/ttH1 :  9/24 bins,  14.0% of template yield smoothed
Zhpt2/ttH2 :  5/24 bins,   2.9% of template yield smoothed
Zhpt2/ttH3 : 15/24 bins,  22.4% of template yield smoothed
Zhpt2/ttZ0 : 20/24 bins,  55.9% of template yield smoothed
Zhpt2/ttZ1 : 21/24 bins,  70.3% of template yield smoothed
Zhpt2/ttZ2 : 15/24 bins,  15.9% of template yield smoothed
Zhpt2/ttZ3 : 24/24 bins, 100.0% of template yield smooth

In [36]:
import numpy as np, awkward as ak, pyarrow.dataset as ds

def sm_weight_spread(path, max_batches=50):
    ws = []
    for i, batch in enumerate(ds.dataset(path, format="parquet").to_batches(
            columns=["events_EFTfitCoefficients", "events_EFTfitCoefficientIndex1",
                     "events_EFTfitCoefficientIndex2"])):
        ch = ak.from_arrow(batch)
        if len(ch) == 0:
            continue
        i1 = ak.to_numpy(ch["events_EFTfitCoefficientIndex1"][0]).astype(int)
        i2 = ak.to_numpy(ch["events_EFTfitCoefficientIndex2"][0]).astype(int)
        k00 = int(np.flatnonzero((i1 == 0) & (i2 == 0))[0])
        ws.append(ak.to_numpy(ch["events_EFTfitCoefficients"])[:, k00])
        if i >= max_batches:
            break
    w = np.concatenate(ws)
    print(f"{path.split('/')[-3]}: N={len(w)}, N_eff/N = {w.sum()**2 / (w**2).sum() / len(w):.3f}")
    print("  SM weight / mean, percentiles 1,10,50,90,99,99.9:",
          np.round(np.percentile(w / w.mean(), [1, 10, 50, 90, 99, 99.9]), 3))

for label, cfg in PROCESSES.items():
    sm_weight_spread(cfg["paths"][0])

ttHSMEFT_2018: N=4512, N_eff/N = 0.207
  SM weight / mean, percentiles 1,10,50,90,99,99.9: [0.    0.    0.028 4.709 7.061 9.128]
ttZSMEFT_2018: N=1029, N_eff/N = 0.213
  SM weight / mean, percentiles 1,10,50,90,99,99.9: [ 0.     0.     0.026  4.578  7.495 10.611]
ttbbSMEFT_2018: N=2578, N_eff/N = 0.182
  SM weight / mean, percentiles 1,10,50,90,99,99.9: [0.000e+00 0.000e+00 1.000e-03 5.366e+00 7.155e+00 8.574e+00]


In [27]:
def bin_response(merged, meta, wc, value=1.0):
    """EFT/SM - 1 per (channel, process, bin), summed over gen-pT bins."""
    acc = {}
    for (ch, proc), full in merged.items():
        m = meta[(ch, proc)]
        if wc not in m[2]:
            continue
        u = unroll(full, ch)[:, :len(m[0])]
        key = (ch, proc.rstrip("0123456789"))
        acc[key] = (acc[key][0] + u, m) if key in acc else (u.copy(), m)
    out = {}
    for key, (u, (i1, i2, names)) in acc.items():
        c = np.array([1.0] + [value if n == wc else 0.0 for n in names[1:]])
        k00 = np.flatnonzero((i1 == 0) & (i2 == 0))[0]
        with np.errstate(divide="ignore", invalid="ignore"):
            out[key] = u @ (c[i1] * c[i2]) / u[:, k00] - 1
    return out

old = np.load("EFT_Parameterization_v7.npy", allow_pickle=True)
new = bin_response(merged, meta, "cbwre")
for (ch, proc), r_new in sorted(new.items()):
    r_old = np.array([old[(proc, f"y2018_{ch}_{b}")].get(("cbW", "cbW"), 0.0)
                      for b in range(len(r_new)) if (proc, f"y2018_{ch}_{b}") in old])
    ok = (r_new[:len(r_old)] > 0) & (r_old > 0)
    ratio = r_old[ok] / r_new[:len(r_old)][ok]
    print(f"{ch} {proc:4}: median old/new = {np.median(ratio):.2f}   "
          f"(16-84%: {np.percentile(ratio, 16):.2f}-{np.percentile(ratio, 84):.2f})")

Zhpt1 ttH : median old/new = 1.47   (16-84%: 0.87-1.87)
Zhpt1 ttZ : median old/new = 1.34   (16-84%: 0.73-1.88)
Zhpt1 tt_B: median old/new = 1.22   (16-84%: 0.72-1.75)
Zhpt2 ttH : median old/new = 1.51   (16-84%: 0.97-1.87)
Zhpt2 ttZ : median old/new = 2.24   (16-84%: 0.89-4.29)
Zhpt2 tt_B: median old/new = 1.19   (16-84%: 0.74-2.82)
Zhpt3 ttH : median old/new = 1.32   (16-84%: 0.79-2.08)
Zhpt3 ttZ : median old/new = 1.32   (16-84%: 0.61-2.89)
Zhpt3 tt_B: median old/new = 1.04   (16-84%: 0.46-3.58)


In [19]:
# ============================================================
# Diagnostics for a Wilson coefficient (run after Cell 1 of eft_scaling.ipynb)
#
# Paste into a notebook cell, or run with:  %run -i diagnose_wc_cell.py
# Uses `merged, meta` from read_inputs() if available (no re-reading needed).
# ============================================================
import json
import numpy as np


def _eval_scaling(entry, point):
    """Per-bin EFT/SM ratio of a JSON entry at a WC point."""
    names = [p.split("[")[0] for p in entry["parameters"]]
    theta = np.array([1.0 if n == "SM" else point.get(n, 0.0) for n in names])
    rows, cols = np.tril_indices(len(theta))
    weights = np.where(rows == cols, 1.0, 2.0) * theta[rows] * theta[cols]
    return np.asarray(entry["scaling"]) @ weights


def _rate_ratio(tot, meta_entry, point):
    """sigma(EFT)/sigma(SM) from summed coefficients."""
    i1, i2, names = meta_entry
    c = np.array([1.0] + [point.get(n, 0.0) for n in names[1:]])
    k00 = np.flatnonzero((i1 == 0) & (i2 == 0))[0]
    return (tot[: len(i1)] @ (c[i1] * c[i2])) / tot[k00]


def _summed_preselection(path):
    """Sum the EFT coefficients over every dumped event in a parquet directory (slow)."""
    import awkward as ak
    import pyarrow.dataset as ds
    cols = ["events_EFTfitCoefficients", "events_EFTfitCoefficientIndex1",
            "events_EFTfitCoefficientIndex2", "events_WCnames"]
    tot, meta_entry = None, None
    for batch in ds.dataset(path, format="parquet").to_batches(columns=cols):
        ch = ak.from_arrow(batch)
        if len(ch) == 0:
            continue
        if meta_entry is None:
            i1 = ak.to_numpy(ch["events_EFTfitCoefficientIndex1"][0]).astype(int)
            i2 = ak.to_numpy(ch["events_EFTfitCoefficientIndex2"][0]).astype(int)
            names = ["SM"] + decode_WCnames(ak.to_list(ch["events_WCnames"][0]))
            meta_entry = (i1, i2, names)
        s = ak.to_numpy(ch["events_EFTfitCoefficients"]).astype(np.float64).sum(axis=0)
        tot = s if tot is None else tot + s
    return tot, meta_entry


def diagnose_wc(wc, value, json_paths=("combine_scaling_2018.json",), merged=None, meta=None,
                read_preselection=False, processes=None, top=10, write_filtered=False):
    """
    wc, value          : WC to test and the value to evaluate at (e.g. its 95% CL bound)
    json_paths         : one or more scaling JSONs to inspect (e.g. nominal and nofallback)
    merged, meta       : output of read_inputs(); gives the rate ratio after the analysis selection
    read_preselection  : also re-read the parquet files for the rate ratio before the selection (slow)
    write_filtered     : write <json>_no_ttB / _ttH_only / _ttZ_only variants of each JSON
    """
    point = {wc: value}
    if isinstance(json_paths, str):
        json_paths = [json_paths]

    # ---- 1. Rate ratio after the analysis selection, from read_inputs() output ----
    if merged is not None and meta is not None:
        print("=" * 72)
        print(f" RATE RATIO AFTER ANALYSIS SELECTION  ({wc} = {value} and 1.0)")
        print("=" * 72)
        by_proc, by_base = {}, {}
        for (ch, proc), full in merged.items():
            m = meta[(ch, proc)]
            if wc not in m[2]:
                continue
            tot = full[..., : len(m[0])].sum(axis=(0, 1))
            for d, key in ((by_proc, proc), (by_base, proc.rstrip("0123456789"))):
                d[key] = (d[key][0] + tot, m) if key in d else (tot, m)
        for label, d in (("per gen-pT process", by_proc), ("per process", by_base)):
            print(f" {label}:")
            for key in sorted(d):
                tot, m = d[key]
                r_v, r_1 = _rate_ratio(tot, m, point), _rate_ratio(tot, m, {wc: 1.0})
                print(f"   {key:8} {wc}={value:<8g}: {r_v:.6f}   {wc}=1: {r_1:.6f}")

    # ---- 2. Rate ratio over all dumped events (preselection only) ----
    if read_preselection:
        processes = processes or globals().get("PROCESSES")
        if processes is None:
            raise ValueError("Pass processes=... or run Cell 1 first (defines PROCESSES)")
        print("\n" + "=" * 72)
        print(" RATE RATIO OVER ALL DUMPED EVENTS (preselection only)")
        print("=" * 72)
        for label, cfg in processes.items():
            for path in cfg["paths"]:
                tot, m = _summed_preselection(path)
                if m is None or wc not in m[2]:
                    print(f"   {label}: no events or {wc} not in sample")
                    continue
                print(f"   {label:8} {wc}={value:<8g}: {_rate_ratio(tot, m, point):.6f}   "
                      f"{wc}=1: {_rate_ratio(tot, m, {wc: 1.0}):.6f}")

    # ---- 3. Most extreme bins in each JSON ----
    for path in json_paths:
        entries = json.load(open(path))
        print("\n" + "=" * 72)
        print(f" MOST EXTREME BINS IN {path} AT {wc} = {value}")
        print("=" * 72)
        rows = []
        for e in entries:
            if wc not in [p.split("[")[0] for p in e["parameters"]]:
                continue
            r = _eval_scaling(e, point)
            b = int(np.argmax(np.abs(r - 1)))
            rows.append((abs(r[b] - 1), e["channel"], e["process"], b, r[b], np.median(r)))
        if not rows:
            print(f"   {wc} not found in this file")
        for dev, ch, proc, b, r, med in sorted(rows, reverse=True)[:top]:
            print(f"   {ch:6} {proc:6} bin {b:2}: ratio = {r:10.4f}   (median over bins: {med:.4f})")

        # ---- 4. Optional filtered JSONs for per-process scans ----
        if write_filtered:
            stem = path.rsplit(".json", 1)[0]
            filters = {"no_ttB":   lambda e: e["process"] != "tt_B",
                       "ttH_only": lambda e: e["process"].startswith("ttH"),
                       "ttZ_only": lambda e: e["process"].startswith("ttZ")}
            for name, keep in filters.items():
                out = f"{stem}_{name}.json"
                json.dump([e for e in entries if keep(e)], open(out, "w"))
                print(f"   wrote {out}")


# ------------------------------------------------------------
# Example call (edit and run). Uses merged/meta from Cell 2 if they exist.
# ------------------------------------------------------------
diagnose_wc(
    "cbwre", 0.15,
    json_paths=["scaling_nofallback.json", "combine_scaling_2018.json"],
    merged=globals().get("merged"), meta=globals().get("meta"),
)

 RATE RATIO AFTER ANALYSIS SELECTION  (cbwre = 0.15 and 1.0)
 per gen-pT process:
   ttH0     cbwre=0.15    : 1.000065   cbwre=1: 1.003041
   ttH1     cbwre=0.15    : 1.000106   cbwre=1: 1.004853
   ttH2     cbwre=0.15    : 1.000192   cbwre=1: 1.008908
   ttH3     cbwre=0.15    : 1.000499   cbwre=1: 1.022600
   ttZ0     cbwre=0.15    : 1.000094   cbwre=1: 1.004856
   ttZ1     cbwre=0.15    : 1.000102   cbwre=1: 1.004167
   ttZ2     cbwre=0.15    : 1.000160   cbwre=1: 1.007330
   ttZ3     cbwre=0.15    : 1.000824   cbwre=1: 1.037327
 per process:
   ttH      cbwre=0.15    : 1.000146   cbwre=1: 1.006719
   ttZ      cbwre=0.15    : 1.000190   cbwre=1: 1.008644

 MOST EXTREME BINS IN scaling_nofallback.json AT cbwre = 0.15
   Zhpt2  ttZ3   bin 23: ratio =   122.1559   (median over bins: 1.0008)
   Zhpt3  ttH1   bin  1: ratio =    63.6299   (median over bins: 1.0002)
   Zhpt3  ttZ2   bin 20: ratio =    43.9321   (median over bins: 1.0005)
   Zhpt3  ttZ1   bin 12: ratio =    27.8504   (media

In [7]:
import numpy as np, awkward as ak, pyarrow.dataset as ds
def inclusive_ratio(path, point):
    cols = ["events_EFTfitCoefficients", "events_EFTfitCoefficientIndex1",
            "events_EFTfitCoefficientIndex2", "events_WCnames"]
    tot, meta = None, None
    for batch in ds.dataset(path, format="parquet").to_batches(columns=cols):
        ch = ak.from_arrow(batch)
        if len(ch) == 0:
            continue
        if meta is None:
            i1 = ak.to_numpy(ch["events_EFTfitCoefficientIndex1"][0]).astype(int)
            i2 = ak.to_numpy(ch["events_EFTfitCoefficientIndex2"][0]).astype(int)
            names = ["SM"] + decode_WCnames(ak.to_list(ch["events_WCnames"][0]))
            meta = (i1, i2, names)
        s = ak.to_numpy(ch["events_EFTfitCoefficients"]).astype(np.float64).sum(axis=0)
        tot = s if tot is None else tot + s
    i1, i2, names = meta
    c = np.array([1.0] + [point.get(n, 0.0) for n in names[1:]])
    k00 = np.flatnonzero((i1 == 0) & (i2 == 0))[0]
    return (tot @ (c[i1] * c[i2])) / tot[k00]

for label, path in [("ttH", f"{BASE}/ttHSMEFT_2018/btag_mask/nominal"),
                    ("ttZ", f"{BASE}/ttZSMEFT_2018/btag_mask/nominal"),
                    ("ttbb", f"{BASE}/ttbbSMEFT_2018/btag_mask/nominal")]:
    for v in [0.15, 1.0]:
        print(label, f"cbwre={v}:", inclusive_ratio(path, {"cbwre": v}))


ttH cbwre=0.15: 1.0000801568741118
ttH cbwre=1.0: 1.0036916663314928
ttZ cbwre=0.15: 1.0001183351042457
ttZ cbwre=1.0: 1.0053863295723517
ttbb cbwre=0.15: 1.0000307346149462
ttbb cbwre=1.0: 1.0050647056886757


In [12]:
import json
d = json.load(open("combine_scaling_2018.json"))
for name, keep in [("no_ttB", lambda e: e["process"] != "tt_B"),
                   ("ttH_only", lambda e: e["process"].startswith("ttH")),
                   ("ttZ_only", lambda e: e["process"].startswith("ttZ"))]:
    json.dump([e for e in d if keep(e)], open(f"scaling_{name}.json", "w"))

In [13]:
d = json.load(open("combine_scaling_2018.json"))
rows = []
for e in d:
    r = eval_scaling(e, {"cbwre": 0.15})
    b = int(np.argmax(np.abs(r - 1)))
    rows.append((abs(r[b] - 1), e["channel"], e["process"], b, r[b]))
for dev, ch, proc, b, r in sorted(rows, reverse=True)[:10]:
    print(f"{ch:6} {proc:6} bin {b:2}: ratio = {r:.3f}")

Zhpt1  ttH3   bin  1: ratio = 1.002
Zhpt2  ttH3   bin  4: ratio = 1.002
Zhpt3  ttH3   bin 21: ratio = 1.002
Zhpt3  ttZ3   bin 17: ratio = 1.001
Zhpt1  ttZ3   bin  0: ratio = 1.001
Zhpt2  ttZ3   bin  0: ratio = 1.001
Zhpt1  ttH2   bin  2: ratio = 1.001
Zhpt2  ttH2   bin 17: ratio = 1.000
Zhpt3  ttZ2   bin  0: ratio = 1.000
Zhpt2  ttH1   bin 17: ratio = 1.000


In [22]:
import numpy as np
fits = np.load("EFT_Parameterization_v7.npy", allow_pickle=True)   # old file name
wc = "cbW"
for key in sorted(fits):
    f = fits[key]
    lin = f.get(("sm", wc), f.get((wc, "sm"), 0.0))
    quad = f.get((wc, wc), 0.0)
    print(f"{str(key):45} lin={lin:+.4f}  quad={quad:+.4f}  r(c=1)-1={lin + quad:+.4f}")

('ttH', 'y2016_Zhpt1_0')                      lin=-0.0000  quad=+0.0032  r(c=1)-1=+0.0032
('ttH', 'y2016_Zhpt1_1')                      lin=+0.0000  quad=+0.0032  r(c=1)-1=+0.0032
('ttH', 'y2016_Zhpt1_10')                     lin=+0.0000  quad=+0.0048  r(c=1)-1=+0.0048
('ttH', 'y2016_Zhpt1_11')                     lin=-0.0000  quad=+0.0046  r(c=1)-1=+0.0046
('ttH', 'y2016_Zhpt1_12')                     lin=-0.0000  quad=+0.0052  r(c=1)-1=+0.0052
('ttH', 'y2016_Zhpt1_13')                     lin=-0.0000  quad=+0.0055  r(c=1)-1=+0.0055
('ttH', 'y2016_Zhpt1_14')                     lin=-0.0000  quad=+0.0060  r(c=1)-1=+0.0060
('ttH', 'y2016_Zhpt1_15')                     lin=+0.0000  quad=+0.0051  r(c=1)-1=+0.0051
('ttH', 'y2016_Zhpt1_16')                     lin=+0.0000  quad=+0.0068  r(c=1)-1=+0.0068
('ttH', 'y2016_Zhpt1_17')                     lin=-0.0000  quad=+0.0060  r(c=1)-1=+0.0060
('ttH', 'y2016_Zhpt1_2')                      lin=+0.0000  quad=+0.0042  r(c=1)-1=+0.0042
('ttH', 'y

In [23]:
def channel_response(merged, meta, wc, value=1.0, by_gen=False):
    """EFT/SM - 1 per (channel, process), summed over the bins in each channel."""
    acc = {}
    for (ch, proc), full in merged.items():
        m = meta[(ch, proc)]
        if wc not in m[2]:
            continue
        key = (ch, proc if by_gen else proc.rstrip("0123456789"))
        tot = full[..., :len(m[0])].sum(axis=(0, 1))
        acc[key] = (acc[key][0] + tot, m) if key in acc else (tot, m)
    for key in sorted(acc):
        tot, m = acc[key]
        print(f"{str(key):25} r(c={value})-1 = {_rate_ratio(tot, m, {wc: value}) - 1:+.5f}")

channel_response(merged, meta, "cbwre")

('Zhpt1', 'ttH')          r(c=1.0)-1 = +0.00348
('Zhpt1', 'ttZ')          r(c=1.0)-1 = +0.00505
('Zhpt2', 'ttH')          r(c=1.0)-1 = +0.00632
('Zhpt2', 'ttZ')          r(c=1.0)-1 = +0.00654
('Zhpt3', 'ttH')          r(c=1.0)-1 = +0.01707
('Zhpt3', 'ttZ')          r(c=1.0)-1 = +0.02596


In [29]:
import ROOT
f = ROOT.TFile.Open("eftdatacards.wc.wp.root")
w = f.Get("w")
print(bool(w.function("r_tt_B_y2018_Zhpt3_0")))   # True -> ttbb was EFT-scaled

TClass::Init:0: RuntimeWarning: no dictionary for class CMSHistErrorPropagator is available
TClass::Init:0: RuntimeWarning: no dictionary for class CMSHistFunc is available
TClass::Init:0: RuntimeWarning: no dictionary for class SimpleGaussianConstraint is available
TClass::Init:0: RuntimeWarning: no dictionary for class ProcessNormalization is available
TClass::Init:0: RuntimeWarning: no dictionary for class CMSHistFuncWrapper is available
TClass::Init:0: RuntimeWarning: no dictionary for class FastHisto is available
TClass::Init:0: RuntimeWarning: no dictionary for class FastTemplate is available
TClass::Init:0: RuntimeWarning: no dictionary for class AsymPow is available
TClass::Init:0: RuntimeWarning: no dictionary for class RooSimultaneousOpt is available


TypeError: none of the 2 overloaded methods succeeded. Full details:
  void* TDirectoryFile::GetObjectChecked(const char* namecycle, const TClass* cl) =>
    TypeError: could not convert argument 2
  abort from C++; program state was reset

Error in <TBufferFile::ReadObject>: trying to read an emulated class (CMSHistErrorPropagator) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying to read an emulated class (SimpleGaussianConstraint) to store in a compiled pointer (TObject)
Error in <TBufferFile::ReadObject>: trying 

In [46]:
import pandas as pd, numpy as np
df = pd.read_pickle("pickled/Nominal2017/TTBar.pkl")
for src in ["hf", "lf", "cferr1", "hfstats1"]:
    print(src, "max |btag_sf - btag_sf%s| =" % src, np.abs(df["btag_sf"] - df[f"btag_sf{src}"]).max())

hf max |btag_sf - btag_sfhf| = 5.3264180804970565
lf max |btag_sf - btag_sflf| = 5.3264180804970565
cferr1 max |btag_sf - btag_sfcferr1| = 5.3264180804970565
hfstats1 max |btag_sf - btag_sfhfstats1| = 5.3264180804970565


In [48]:
import pandas as pd, numpy as np
df = pd.read_pickle("pickled/Nominal2017/TTBar.pkl")
srcs = ["hf", "lf", "cferr1", "cferr2", "hfstats1", "hfstats2", "lfstats1", "lfstats2"]

print(df[["btag_sf", "btag_sfhf", "btag_sfhf_up", "btag_sfhf_down"]].describe().T[["mean", "min", "max"]])
print("all source centrals identical:",
      all(np.allclose(df["btag_sfhf"], df[f"btag_sf{s}"]) for s in srcs))
print("fraction of btag_sfhf == 1:", np.mean(df["btag_sfhf"] == 1.0))

pct = [1, 16, 50, 84, 99]
print("hf_up / btag_sfhf percentiles:", np.round(np.percentile(df["btag_sfhf_up"] / df["btag_sfhf"], pct), 3))
print("hf_up / btag_sf   percentiles:", np.round(np.percentile(df["btag_sfhf_up"] / df["btag_sf"], pct), 3))


                    mean       min        max
btag_sf         0.960847  0.304871   6.326418
btag_sfhf       1.000000  1.000000   1.000000
btag_sfhf_up    0.919709  0.044551   3.754339
btag_sfhf_down  0.991502  0.372769  20.788842
all source centrals identical: True
fraction of btag_sfhf == 1: 1.0
hf_up / btag_sfhf percentiles: [0.572 0.729 0.883 1.111 1.591]
hf_up / btag_sf   percentiles: [0.587 0.95  1.01  1.022 1.037]
